In [1]:
# ══════════════════════════════════════════════════════════════
# MASTER SETUP CELL — Run this FIRST every time you open Colab
# If you see any NameError, it means you forgot to run this cell
# ══════════════════════════════════════════════════════════════
from google.colab import drive
import os, torch

drive.mount("/content/drive")

BASE   = "/content/drive/MyDrive/Phishing_Project"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(f"{BASE}/Data",   exist_ok=True)
os.makedirs(f"{BASE}/Models", exist_ok=True)

print(f"✅ Drive mounted")
print(f"✅ BASE = {BASE}")
print(f"✅ Device = {device}")

Mounted at /content/drive
✅ Drive mounted
✅ BASE = /content/drive/MyDrive/Phishing_Project
✅ Device = cuda


In [ ]:
# ── PASTE INTO COLAB CELL 1-A ──────────────────────────────────────
# Environment: Google Colab
# Time: ~10 minutes on first run

!pip install -q transformers datasets accelerate
!pip install -q scikit-learn shap lime tldextract
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q mlflow fastapi uvicorn nest-asyncio pyngrok
!pip install -q kaggle huggingface_hub


print("✅ All libraries installed.")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import joblib
# Load lexical model
lexical_clf = joblib.load(f"{BASE}/Models/lexical_rf.pkl")
print("✅ Lexical model loaded")

# Load linguistic model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer_ling = AutoTokenizer.from_pretrained(f"{BASE}/Models/ling_model")
model_ling = AutoModelForSequenceClassification.from_pretrained(
    f"{BASE}/Models/ling_model"
).to(device)
model_ling.eval()
print("✅ Linguistic model loaded")

# Load CLIP
import clip
clip_model, clip_prep = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("✅ CLIP loaded")

✅ Lexical model loaded


Loading weights:   0%|          | 0/104 [00:02<?, ?it/s]

✅ Linguistic model loaded


100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 81.4MiB/s]


✅ CLIP loaded


In [ ]:
# ── PASTE INTO COLAB CELL 2-A ──────────────────────────────────────
# Environment: Google Colab
# Time: instant (just defines a function)

import math, re
from urllib.parse import urlparse
import tldextract

def extract_url_features(url: str) -> dict:
    url = str(url)
    try:
        parsed = urlparse(url)
        ext    = tldextract.extract(url)
        host   = parsed.hostname or ""
        path   = parsed.path or ""
        chars  = set(url)
        entropy = -sum(
            (url.count(c)/len(url)) * math.log2(url.count(c)/len(url))
            for c in chars if url.count(c) > 0
        ) if len(url) > 0 else 0
        return {
            "url_length":      len(url),
            "hostname_length": len(host),
            "num_dots":        url.count("."),
            "has_at_symbol":   int("@" in url),
            "has_ip":          int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host))),
            "subdomain_depth": len(ext.subdomain.split(".")) if ext.subdomain else 0,
            "entropy":         round(entropy, 4),
            "is_https":        int(parsed.scheme == "https"),
            "path_length":     len(path),
            "num_hyphens":     url.count("-"),
        }
    except Exception:
        return {k: 0 for k in ["url_length","hostname_length","num_dots",
                               "has_at_symbol","has_ip","subdomain_depth",
                               "entropy","is_https","path_length","num_hyphens"]}

# Sanity check
print(extract_url_features("http://paypa1-secure.xyz/login?user=@victim"))



{'url_length': 43, 'hostname_length': 17, 'num_dots': 1, 'has_at_symbol': 1, 'has_ip': 0, 'subdomain_depth': 0, 'entropy': 4.5943, 'is_https': 0, 'path_length': 6, 'num_hyphens': 1}


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — MASTER SETUP (Run this EVERY time you open Colab)
# This defines BASE and all variables needed by every other cell
# ══════════════════════════════════════════════════════════════════

# Step 1: Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Step 2: Define BASE — every other cell uses this variable
import os, torch, joblib
BASE   = "/content/drive/MyDrive/Phishing_Project"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 3: Create folders if they don't exist yet
os.makedirs(f"{BASE}/Data",   exist_ok=True)
os.makedirs(f"{BASE}/Models", exist_ok=True)

# Step 4: Auto-reload saved models if they exist on Drive
# (so you don't need to retrain after a session restart)
lexical_clf = None
tokenizer_ling = None
model_ling = None

if os.path.exists(f"{BASE}/Models/lexical_rf.pkl"):
    lexical_clf = joblib.load(f"{BASE}/Models/lexical_rf.pkl")
    print("✅ Lexical model loaded from Drive")
else:
    print("⚠️  Lexical model not found — run Phase 2 to train it")

if os.path.exists(f"{BASE}/Models/ling_model/config.json"):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer_ling = AutoTokenizer.from_pretrained(f"{BASE}/Models/ling_model")
    model_ling = AutoModelForSequenceClassification.from_pretrained(
        f"{BASE}/Models/ling_model"
    ).to(device)
    model_ling.eval()
    print("✅ Linguistic model loaded from Drive")
else:
    print("⚠️  Linguistic model not found — run Phase 3 to train it")

# Step 5: Confirm everything is ready
print(f"\n{'='*50}")
print(f"  BASE   = {BASE}")
print(f"  device = {device}")
print(f"  Lexical model ready : {lexical_clf is not None}")
print(f"  Linguistic model ready: {model_ling is not None}")
print(f"{'='*50}")
print("✅ Setup complete. You can now run any phase.")

Mounted at /content/drive
✅ Lexical model loaded from Drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Linguistic model loaded from Drive

  BASE   = /content/drive/MyDrive/Phishing_Project
  device = cuda
  Lexical model ready : True
  Linguistic model ready: True
✅ Setup complete. You can now run any phase.


In [ ]:
# ── PASTE INTO COLAB CELL 1-B ──────────────────────────────────────
# Environment: Google Colab
# Time: instant

from google.colab import drive
import os, json

drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/Phishing_Project"
os.makedirs(f"{BASE}/Data",   exist_ok=True)
os.makedirs(f"{BASE}/Models", exist_ok=True)

# ── Kaggle credentials ──────────────────────────────────────────────
# Paste the values from your kaggle.json file below
KAGGLE_USERNAME = "sai7766"   # ← REPLACE
KAGGLE_KEY      = "KGAT_4107335b30584d1e4c89fd235ff50490"        # ← REPLACE

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("✅ Drive mounted. API key set.")
print("BASE path:", BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. API key set.
BASE path: /content/drive/MyDrive/Phishing_Project


In [ ]:
!pip install datasets


In [ ]:
!pip install "datasets<3.0.0"
# Restart your session after running this line


In [ ]:


BASE = "/content/drive/MyDrive" # This sets your Drive as the base folder


In [ ]:
import pandas as pd
from datasets import load_dataset
import os

# Ensure BASE is defined (e.g., your Google Drive mount point)
# BASE = "/content/drive/MyDrive"

print("Downloading phishing email dataset from HuggingFace...")
# Loading JSON directly to bypass the broken/unsupported loading script
phishing_ds = load_dataset(
    "json",
    data_files="hf://datasets/ealvaradob/phishing-dataset/combined_reduced.json",
    split="train"
)
phishing_df = phishing_ds.to_pandas()[["text", "label"]]
# Filter for phishing (label 1) and remap to class 2
phishing_df = phishing_df[phishing_df["label"] == 1].copy()
phishing_df["label"] = 2

print("Downloading Enron spam/ham dataset from HuggingFace...")
enron_ds = load_dataset("SetFit/enron_spam", split="train")
enron_df = enron_ds.to_pandas()[["text", "label"]]
# Enron: 0 = Ham, 1 = Spam
enron_df["label"] = enron_df["label"].map({0: 0, 1: 1})

# Combine datasets
email_df = pd.concat([phishing_df, enron_df], ignore_index=True)
email_df["text"] = email_df["text"].astype(str).str.lower().str.strip()
email_df = email_df.dropna()

# Save logic
os.makedirs(f"{BASE}/Data", exist_ok=True)
save_path = f"{BASE}/Data/master_emails.csv"
email_df.to_csv(save_path, index=False)

print(f"✅ Saved to {save_path}")
print("Label counts:", email_df["label"].value_counts().to_dict())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/31716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Saved to /content/drive/MyDrive/Phishing_Project/Data/master_emails.csv
Label counts: {2: 32702, 1: 16163, 0: 15553}


In [ ]:
# ── PASTE INTO COLAB CELL 1-D ──────────────────────────────────────
# Environment: Google Colab
# Time: ~5 minutes (downloads ~40 MB, unzips to ~150 MB)
# SAVES TO: BASE/Data/malicious_phish.csv

import subprocess

print("Downloading URL dataset from Kaggle...")
result = subprocess.run([
    "kaggle", "datasets", "download",
    "-d", "sid321axn/malicious-urls-dataset",
    "-p", f"{BASE}/Data", "--unzip"
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print(f"✅ URL dataset saved to {BASE}/Data/")
    url_df = pd.read_csv(f"{BASE}/Data/malicious_phish.csv")
    print("Rows:", len(url_df), "| Columns:", url_df.columns.tolist())



Dataset URL: https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset
License(s): CC0-1.0


✅ URL dataset saved to /content/drive/MyDrive/Phishing_Project/Data/
Rows: 651191 | Columns: ['url', 'type']


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2-B (FIXED) — Lexical Engine with Class Imbalance Fix
# Environment: Google Colab
# Time: ~10 minutes
# SAVES TO: BASE/Models/lexical_rf.pkl
#           BASE/Models/url_feature_names.pkl
# ══════════════════════════════════════════════════════════════════

import pandas as pd, numpy as np, joblib, math, re
from urllib.parse import urlparse
import tldextract
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
import os # Import os module to create directories

# ── FIX 1: Better label mapping ──────────────────────────────────
# defacement is NOT safe — treat it as malicious (1)
url_df = pd.read_csv(f"{BASE}/Data/malicious_phish.csv")
url_df["label"] = url_df["type"].map({
    "benign":      0,   # Safe
    "defacement":  1,   # Malicious (was wrongly mapped to 0 before)
    "phishing":    1,   # Malicious
    "malware":     1,   # Malicious
})
url_df = url_df[["url", "label"]].dropna()
print("Label distribution BEFORE balancing:")
print(url_df["label"].value_counts())

# ── FIX 2: Expanded feature set (18 features instead of 10) ──────
def extract_url_features(url: str) -> dict:
    url = str(url).strip()
    try:
        parsed = urlparse(url)
        ext    = tldextract.extract(url)
        host   = parsed.hostname or ""
        path   = parsed.path or ""
        query  = parsed.query or ""

        # Shannon entropy
        chars = set(url)
        entropy = -sum(
            (url.count(c)/len(url)) * math.log2(url.count(c)/len(url))
            for c in chars if url.count(c) > 0
        ) if len(url) > 0 else 0

        # Suspicious keyword count
        SUSPICIOUS_WORDS = [
            "login","verify","secure","update","account","banking",
            "confirm","password","signin","paypal","ebay","amazon",
            "apple","microsoft","support","service","free","lucky"
        ]
        suspicious_count = sum(1 for w in SUSPICIOUS_WORDS if w in url.lower())

        # Digit ratio in hostname
        digit_ratio = sum(c.isdigit() for c in host) / len(host) if host else 0

        return {
            # Original 10 features
            "url_length":       len(url),
            "hostname_length":  len(host),
            "num_dots":         url.count("."),
            "has_at_symbol":    int("@" in url),
            "has_ip":           int(bool(re.match(
                                    r"^\d{1,3}(\.\d{1,3}){3}$", host))),
            "subdomain_depth":  len(ext.subdomain.split(".")) if ext.subdomain else 0,
            "entropy":          round(entropy, 4),
            "is_https":         int(parsed.scheme == "https"),
            "path_length":      len(path),
            "num_hyphens":      url.count("-"),
            # NEW features that significantly boost Malicious recall
            "num_slashes":      url.count("/"),
            "num_query_params": len(query.split("&")) if query else 0,
            "has_port":         int(bool(parsed.port)),
            "tld_suspicious":   int(ext.suffix in [
                                    "xyz","tk","ml","ga","cf","gq",
                                    "top","click","link","work","party"]),
            "suspicious_words": suspicious_count,
            "digit_ratio_host": round(digit_ratio, 4),
            "url_has_redirect": int("redirect" in url.lower() or "url=" in url.lower()),
            "double_slash_path":int("//" in path),
        }
    except Exception:
        return {k: 0 for k in [
            "url_length","hostname_length","num_dots","has_at_symbol","has_ip",
            "subdomain_depth","entropy","is_https","path_length","num_hyphens",
            "num_slashes","num_query_params","has_port","tld_suspicious",
            "suspicious_words","digit_ratio_host","url_has_redirect","double_slash_path"
        ]}

# ── FIX 3: Balance classes via oversampling ───────────────────────
# Oversample the minority Malicious class to match Safe count
safe_df      = url_df[url_df["label"] == 0]
malicious_df = url_df[url_df["label"] == 1]

malicious_upsampled = resample(
    malicious_df,
    replace=True,                    # sample with replacement
    n_samples=len(safe_df),          # match majority class size
    random_state=42
)
url_balanced = pd.concat([safe_df, malicious_upsampled]).sample(
    frac=1, random_state=42          # shuffle
).reset_index(drop=True)

print("\nLabel distribution AFTER balancing:")
print(url_balanced["label"].value_counts())

# ── Extract features ──────────────────────────────────────────────
print("\nExtracting URL features... (~8 min)")
X = pd.DataFrame([extract_url_features(u) for u in url_balanced["url"]])
y = url_balanced["label"].reset_index(drop=True)

# ── FIX 4: Use 0.2 test split (more training data) ───────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── FIX 5: class_weight="balanced" + more estimators ─────────────
print("Training Random Forest (300 trees, balanced weights)...")
lexical_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight="balanced",         # penalises misclassifying minority class
    n_jobs=-1,
    random_state=42
)
lexical_clf.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────
y_pred = lexical_clf.predict(X_test)
f1 = f1_score(y_test, y_pred, average="weighted")

print("\n📊 Results:")
print(classification_report(y_test, y_pred, target_names=["Safe", "Malicious"]))
print(f"✅ Weighted F1: {f1:.4f}  (target > 0.97)")

# ── Save ──────────────────────────────────────────────────────────
os.makedirs(f"{BASE}/Models", exist_ok=True) # Ensure Models directory exists
joblib.dump(lexical_clf,           f"{BASE}/Models/lexical_rf.pkl")
joblib.dump(X_train.columns.tolist(), f"{BASE}/Models/url_feature_names.pkl")
print(f"✅ Saved to {BASE}/Models/")

Label distribution BEFORE balancing:
label
0    428103
1    223088
Name: count, dtype: int64

Label distribution AFTER balancing:
label
0    428103
1    428103
Name: count, dtype: int64

Extracting URL features... (~8 min)
Training Random Forest (300 trees, balanced weights)...

📊 Results:
              precision    recall  f1-score   support

        Safe       0.97      0.96      0.96     85621
   Malicious       0.96      0.97      0.96     85621

    accuracy                           0.96    171242
   macro avg       0.96      0.96      0.96    171242
weighted avg       0.96      0.96      0.96    171242

✅ Weighted F1: 0.9629  (target > 0.97)
✅ Saved to /content/drive/MyDrive/Phishing_Project/Models/


In [ ]:
import os

# Check what files actually exist in your Data folder
data_path = f"{BASE}/Data"
files = os.listdir(data_path)
print("Files in Data folder:", files)

Files in Data folder: ['master_emails.csv', 'malicious_phish.csv']


In [ ]:
import pandas as pd, torch, os, warnings
from transformers import DistilBertTokenizerFast
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Running on: {device}")  # Must print: cuda

# Load data
BASE = "/content/drive/MyDrive/Phishing_Project"
email_df = pd.read_csv(f"{BASE}/Data/master_emails.csv")
print(f"Raw dataset: {len(email_df)} rows")
print("Raw label counts:", email_df["label"].value_counts().to_dict())

# Balance — equal samples per class (no * 3 multiplier)
min_count = email_df["label"].value_counts().min()
balanced_df = (
    email_df
    .groupby("label", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), min_count), random_state=42))
    .reset_index(drop=True)
    .sample(frac=1, random_state=42)
)
print(f"\nBalanced label counts:")
print(balanced_df["label"].value_counts().to_string())

# Tokenizer
TOKENIZER_NAME = "distilbert-base-uncased"
tok = DistilBertTokenizerFast.from_pretrained(TOKENIZER_NAME)

# Dataset class
class EmailDS(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts    = [str(t) for t in texts]
        self.labels   = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels":         torch.tensor(self.labels[i], dtype=torch.long)
        }

# Split
tr_txt, vl_txt, tr_lbl, vl_lbl = train_test_split(
    balanced_df["text"].tolist(),
    balanced_df["label"].tolist(),
    test_size=0.1,
    stratify=balanced_df["label"].tolist(),
    random_state=42
)

train_ds = EmailDS(tr_txt, tr_lbl, tok)
val_ds   = EmailDS(vl_txt, vl_lbl, tok)

print(f"\n✅ Train: {len(train_ds)} | Val: {len(val_ds)}")
print("✅ Cell 3-A complete. Ready for Cell 3-B (DistilBERT training).")

🚀 Running on: cuda


Raw dataset: 64418 rows
Raw label counts: {2: 32702, 1: 16163, 0: 15553}

Balanced label counts:
label
1    15553
0    15553
2    15553

✅ Train: 41993 | Val: 4666
✅ Cell 3-A complete. Ready for Cell 3-B (DistilBERT training).


In [ ]:
from transformers import (
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score
import numpy as np, os

model_ling = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3  # 0=Safe 1=Spam 2=Phishing
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds, average="weighted")}

LING_MODEL_SAVE = f"{BASE}/Models/ling_model"

args = TrainingArguments(
    output_dir=LING_MODEL_SAVE,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    fp16=True,  # Half-precision — faster on T4
    learning_rate=2e-5,
    warmup_ratio=0.06,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model_ling,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("🚀 Training started... (~45–60 min)")
trainer.train()

# Save best model checkpoint to Drive
trainer.save_model(LING_MODEL_SAVE)
tok.save_pretrained(LING_MODEL_SAVE)

print(f"✅ Linguistic engine saved to: {LING_MODEL_SAVE}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 Training started... (~45–60 min)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Linguistic engine saved to: /content/drive/MyDrive/Phishing_Project/Models/ling_model


In [ ]:
# ── PASTE INTO COLAB CELL 3-C ──────────────────────────────────────
# USE THIS if you are starting a fresh Colab session after training
# (Models saved to Drive persist across sessions)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

LING_MODEL_SAVE = f"{BASE}/Models/ling_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tok = AutoTokenizer.from_pretrained(LING_MODEL_SAVE)
model_ling = AutoModelForSequenceClassification.from_pretrained(LING_MODEL_SAVE)

model_ling.eval().to(device)

LABELS = ["safe", "spam", "phishing"]

def predict_email(text: str) -> dict:
    inputs = tok(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        probs = torch.softmax(model_ling(**inputs).logits, dim=-1)[0].tolist()

    return {LABELS[i]: round(probs[i], 4) for i in range(3)}

# Quick test
print(predict_email("URGENT: Verify your account now at http://secure-fake.xyz"))
print(predict_email("Are you free for coffee at 3pm tomorrow?"))

print("✅ Linguistic engine loaded and ready.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'safe': 0.0008, 'spam': 0.872, 'phishing': 0.1273}
{'safe': 0.0227, 'spam': 0.0056, 'phishing': 0.9717}
✅ Linguistic engine loaded and ready.


In [ ]:
# ── PASTE INTO COLAB CELL 4-A ──────────────────────────────────────
# Environment: Google Colab (GPU)
# Time: ~2–3 minutes (downloads CLIP weights ~350 MB)
# SAVES TO: nothing (CLIP weights cached in Colab /root/.cache/)

import clip, torch
from PIL import Image

clip_model, clip_prep = clip.load("ViT-B/32", device=device)
clip_model.eval()

print(f"✅ CLIP loaded on {device}")

def predict_logo(image_path: str, brand_name: str) -> dict:
    """Score how closely an image matches a claimed brand logo."""
    try:
        img = Image.open(image_path).convert("RGB")
        image = clip_prep(img).unsqueeze(0).to(device)

        prompts = clip.tokenize([
            f"official {brand_name} logo",
            "unrelated logo or random graphic",
        ]).to(device)

        with torch.no_grad():
            i_feat = clip_model.encode_image(image)
            t_feat = clip_model.encode_text(prompts)

            i_feat = i_feat / i_feat.norm(dim=-1, keepdim=True)
            t_feat = t_feat / t_feat.norm(dim=-1, keepdim=True)

            sim = (i_feat @ t_feat.T).squeeze()
            m = sim[0].item()

        return {
            "brand_match_score": round(m, 4),
            "visual_phishing_score": round(1.0 - m, 4),
            "is_impersonation": m < 0.5
        }

    except FileNotFoundError:
        return {
            "brand_match_score": None,
            "visual_phishing_score": 0.0,
            "is_impersonation": False
        }

print("✅ Visual Engine ready. Use: predict_logo(path, brand_name)")

100%|███████████████████████████████████████| 338M/338M [00:08<00:00, 42.4MiB/s]


✅ CLIP loaded on cuda
✅ Visual Engine ready. Use: predict_logo(path, brand_name)


In [ ]:
#vironment: Google Colab
# Time: instant

def fuse_scores(s_ling=None, s_lex=None, s_vis=None,
                spf_fail=False, dkim_fail=False, blacklisted=False) -> dict:
    W = {"ling": 0.40, "lex": 0.35, "vis": 0.25}

    # Hard overrides — fire before soft ensemble
    if spf_fail and dkim_fail and blacklisted:
        return {"verdict":"Phishing","score":1.0,"override":True,"active_engines":["hard_rules"]}
    if s_vis is not None and s_vis > 0.85:
        return {"verdict":"Phishing","score":1.0,"override":True,"active_engines":["visual"]}

    # Soft weighted ensemble
    scores  = {k:v for k,v in [("ling",s_ling),("lex",s_lex),("vis",s_vis)] if v is not None}
    if not scores:
        return {"verdict":"Unknown","score":0.0,"override":False,"active_engines":[]}
    total_w = sum(W[k] for k in scores)
    s_total = sum(scores[k]*W[k]/total_w for k in scores)

    verdict = "Phishing" if s_total>=0.65 else ("Scam" if s_total>=0.35 else "Genuine")
    return {"verdict":verdict,"score":round(s_total,4),"override":False,"active_engines":list(scores.keys())}

# Tests
print(fuse_scores(0.92, 0.95, 0.88))     # → Phishing
print(fuse_scores(0.05, 0.08))            # → Genuine
print(fuse_scores(0.3, spf_fail=True, dkim_fail=True, blacklisted=True))  # → override


{'verdict': 'Phishing', 'score': 1.0, 'override': True, 'active_engines': ['visual']}
{'verdict': 'Genuine', 'score': 0.064, 'override': False, 'active_engines': ['ling', 'lex']}
{'verdict': 'Phishing', 'score': 1.0, 'override': True, 'active_engines': ['hard_rules']}


In [ ]:
# ── PASTE INTO COLAB CELL 6-A ──────────────────────────────────────
# Environment: Google Colab
# Time: instant

import numpy as np

def generate_explanation(shap_vals=None, lime_vals=None,
                         clip_result=None, flags=None) -> dict:
    feats = []
    if shap_vals:
        for tok_name, score in sorted(shap_vals, key=lambda x:abs(x[1]), reverse=True)[:3]:
            if abs(score) > 0.05:
                feats.append(f'Suspicious keyword: "{tok_name}" (impact={score:+.2f})')
    if lime_vals:
        for feat, score in sorted(lime_vals, key=lambda x:abs(x[1]), reverse=True)[:2]:
            if score > 0: feats.append(f"Suspicious URL pattern: {feat}")
    if clip_result and clip_result.get("brand_match_score") is not None:
        bms = clip_result["brand_match_score"]
        if bms < 0.5: feats.append(f"Logo impersonation: similarity={bms:.2f}")
    if flags:
        for flag in flags: feats.append(f"Auth failure: {flag}")
    return {"top_features": feats[:5]}


def full_threat_scan(email_text=None, url_input=None, image_path=None,
                     brand_to_check=None, spf_fail=False, dkim_fail=False,
                     verbose=True) -> dict:
    s_ling = s_lex = s_vis = None
    clip_result = lime_vals = shap_vals = None
    flags = []

    if email_text:
        lr = predict_email(email_text)
        s_ling = min(lr["phishing"] + 0.5*lr["spam"], 1.0)

    if url_input:
        feats  = np.array(list(extract_url_features(url_input).values()))
        s_lex  = float(lexical_clf.predict_proba([feats])[0][1])

    if image_path and brand_to_check:
        clip_result = predict_logo(image_path, brand_to_check)
        s_vis = clip_result["visual_phishing_score"]

    if spf_fail:  flags.append("SPF_FAIL")
    if dkim_fail: flags.append("DKIM_FAIL")

    verdict     = fuse_scores(s_ling, s_lex, s_vis, spf_fail, dkim_fail)
    explanation = generate_explanation(shap_vals, lime_vals, clip_result, flags)

    score = verdict["score"]
    conf  = "High" if score>=0.80 or verdict["override"] else ("Medium" if score>=0.50 else "Low")

    result = {**verdict, "confidence":conf,
"scores":{"linguistic":round(s_ling,4) if s_ling else None,
"lexical":round(s_lex,4) if s_lex else None,
"visual":round(s_vis,4) if s_vis else None},
"flags":flags, "explanation":explanation}

    if verbose:
        v = result["verdict"]
        icon = "🚨" if v=="Phishing" else ("⚠️" if v=="Scam" else "✅")
        print("\n" + "═"*55)
        print(f"  {icon}  VERDICT: {v.upper()}  [{conf} — {score*100:.1f}%]")
        print("═"*55)
        for eng, sc in result["scores"].items():
            if sc: print(f"  {eng.capitalize():15s}: {sc:.4f}  {'█'*int(sc*20)}")
        if flags: print(f"\n  Flags: {chr(44).join(flags)}")
        if explanation["top_features"]:
            print("\n  Evidence:")
            for f in explanation["top_features"]: print(f"    • {f}")
        print("═"*55)

    return result

print("✅ Full pipeline ready: full_threat_scan()")



✅ Full pipeline ready: full_threat_scan()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# RECOVERY CELL — Run this if you get NameError in Phase 7
# This redefines ALL functions lost after session restart
# ══════════════════════════════════════════════════════════════════
import os, math, re, torch, joblib, numpy as np, warnings
from urllib.parse import urlparse
import tldextract
import warnings
warnings.filterwarnings("ignore")

BASE   = "/content/drive/MyDrive/Phishing_Project"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 1. Reload extract_url_features ───────────────────────────────
def extract_url_features(url: str) -> dict:
    url = str(url).strip()
    try:
        parsed = urlparse(url)
        ext    = tldextract.extract(url)
        host   = parsed.hostname or ""
        path   = parsed.path or ""
        query  = parsed.query or ""
        chars  = set(url)
        entropy = -sum(
            (url.count(c)/len(url)) * math.log2(url.count(c)/len(url))
            for c in chars if url.count(c) > 0
        ) if len(url) > 0 else 0
        SUSPICIOUS_WORDS = [
            "login","verify","secure","update","account","banking",
            "confirm","password","signin","paypal","ebay","amazon",
            "apple","microsoft","support","service","free","lucky"
        ]
        suspicious_count = sum(1 for w in SUSPICIOUS_WORDS if w in url.lower())
        digit_ratio = sum(c.isdigit() for c in host) / len(host) if host else 0
        return {
            "url_length":        len(url),
            "hostname_length":   len(host),
            "num_dots":          url.count("."),
            "has_at_symbol":     int("@" in url),
            "has_ip":            int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host))),
            "subdomain_depth":   len(ext.subdomain.split(".")) if ext.subdomain else 0,
            "entropy":           round(entropy, 4),
            "is_https":          int(parsed.scheme == "https"),
            "path_length":       len(path),
            "num_hyphens":       url.count("-"),
            "num_slashes":       url.count("/"),
            "num_query_params":  len(query.split("&")) if query else 0,
            "has_port":          int(bool(parsed.port)),
            "tld_suspicious":    int(ext.suffix in [
                                     "xyz","tk","ml","ga","cf","gq",
                                     "top","click","link","work","party"]),
            "suspicious_words":  suspicious_count,
            "digit_ratio_host":  round(digit_ratio, 4),
            "url_has_redirect":  int("redirect" in url.lower() or "url=" in url.lower()),
            "double_slash_path": int("//" in path),
        }
    except Exception:
        return {k: 0 for k in [
            "url_length","hostname_length","num_dots","has_at_symbol","has_ip",
            "subdomain_depth","entropy","is_https","path_length","num_hyphens",
            "num_slashes","num_query_params","has_port","tld_suspicious",
            "suspicious_words","digit_ratio_host","url_has_redirect","double_slash_path"
        ]}

print("✅ extract_url_features defined")

# ── 2. Reload lexical model from Drive ───────────────────────────
lexical_clf = joblib.load(f"{BASE}/Models/lexical_rf.pkl")
print("✅ lexical_clf loaded")

# ── 3. Reload linguistic model from Drive ────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer_ling = AutoTokenizer.from_pretrained(f"{BASE}/Models/ling_model")
model_ling = AutoModelForSequenceClassification.from_pretrained(
    f"{BASE}/Models/ling_model"
).to(device)
model_ling.eval()
print("✅ model_ling loaded")

# ── 4. Redefine predict_email ─────────────────────────────────────
LABELS = ["safe", "spam", "phishing"]
def predict_email(text: str) -> dict:
    inputs = tokenizer_ling(
        text, return_tensors="pt", truncation=True, max_length=256
    ).to(device)
    with torch.no_grad():
        probs = torch.softmax(model_ling(**inputs).logits, dim=-1)[0].tolist()
    return {LABELS[i]: round(probs[i], 4) for i in range(3)}

print("✅ predict_email defined")

# ── 5. Redefine fuse_scores ───────────────────────────────────────
def fuse_scores(s_ling=None, s_lex=None, s_vis=None,
                spf_fail=False, dkim_fail=False, blacklisted=False) -> dict:
    W = {"ling": 0.40, "lex": 0.35, "vis": 0.25}
    if spf_fail and dkim_fail and blacklisted:
        return {"verdict":"Phishing","score":1.0,"override":True,"active_engines":["hard_rules"]}
    if s_vis is not None and s_vis > 0.85:
        return {"verdict":"Phishing","score":1.0,"override":True,"active_engines":["visual"]}
    scores  = {k:v for k,v in [("ling",s_ling),("lex",s_lex),("vis",s_vis)] if v is not None}
    if not scores:
        return {"verdict":"Unknown","score":0.0,"override":False,"active_engines":[]}
    total_w = sum(W[k] for k in scores)
    s_total = sum(scores[k]*W[k]/total_w for k in scores)
    verdict = "Phishing" if s_total>=0.65 else ("Scam" if s_total>=0.35 else "Genuine")
    return {"verdict":verdict,"score":round(s_total,4),"override":False,
            "active_engines":list(scores.keys())}

print("✅ fuse_scores defined")

# ── 6. Redefine generate_explanation ─────────────────────────────
def generate_explanation(shap_vals=None, lime_vals=None,
                         clip_result=None, flags=None) -> dict:
    feats = []
    if shap_vals:
        for tok_name, score in sorted(shap_vals, key=lambda x:abs(x[1]), reverse=True)[:3]:
            if abs(score) > 0.05:
                feats.append(f'Suspicious keyword: "{tok_name}" (impact={score:+.2f})')
    if lime_vals:
        for feat, score in sorted(lime_vals, key=lambda x:abs(x[1]), reverse=True)[:2]:
            if score > 0: feats.append(f"Suspicious URL pattern: {feat}")
    if clip_result and clip_result.get("brand_match_score") is not None:
        bms = clip_result["brand_match_score"]
        if bms < 0.5: feats.append(f"Logo impersonation: similarity={bms:.2f}")
    if flags:
        for flag in flags: feats.append(f"Auth failure: {flag}")
    return {"top_features": feats[:5]}

print("✅ generate_explanation defined")

# ── 7. Redefine full_threat_scan ──────────────────────────────────
def full_threat_scan(email_text=None, url_input=None, image_path=None,
                     brand_to_check=None, spf_fail=False, dkim_fail=False,
                     verbose=True) -> dict:
    s_ling = s_lex = s_vis = None
    clip_result = None
    flags = []

    if email_text:
        lr = predict_email(email_text)
        s_ling = min(lr["phishing"] + 0.5 * lr["spam"], 1.0)

    if url_input:
        feats = np.array(list(extract_url_features(url_input).values()))
        s_lex = float(lexical_clf.predict_proba([feats])[0][1])

    if image_path and brand_to_check:
        clip_result = predict_logo(image_path, brand_to_check)
        s_vis = clip_result["visual_phishing_score"]

    if spf_fail:  flags.append("SPF_FAIL")
    if dkim_fail: flags.append("DKIM_FAIL")

    verdict     = fuse_scores(s_ling, s_lex, s_vis, spf_fail, dkim_fail)
    explanation = generate_explanation(None, None, clip_result, flags)

    score = verdict["score"]
    conf  = "High" if score>=0.80 or verdict["override"] else (
            "Medium" if score>=0.50 else "Low")

    result = {
        **verdict, "confidence": conf,
        "scores": {
            "linguistic": round(s_ling, 4) if s_ling is not None else None,
            "lexical":    round(s_lex,  4) if s_lex  is not None else None,
            "visual":     round(s_vis,  4) if s_vis  is not None else None,
        },
        "flags": flags,
        "explanation": explanation
    }

    if verbose:
        v    = result["verdict"]
        icon = "🚨" if v=="Phishing" else ("⚠️" if v=="Scam" else "✅")
        print("\n" + "═"*55)
        print(f"  {icon}  VERDICT: {v.upper()}  [{conf} — {score*100:.1f}%]")
        print("═"*55)
        for eng, sc in result["scores"].items():
            if sc is not None:
                bar = "█" * int(sc * 20)
                print(f"  {eng.capitalize():15s}: {sc:.4f}  {bar}")
        if flags:
            print(f"\n  Flags: {', '.join(flags)}")
        if explanation["top_features"]:
            print("\n  Evidence:")
            for f in explanation["top_features"]:
                print(f"    • {f}")
        print("═"*55)

    return result

print("✅ full_threat_scan defined")
print("\n" + "="*55)
print("  ✅ ALL FUNCTIONS RESTORED — Ready for Phase 7 tests")
print("="*55)

✅ extract_url_features defined
✅ lexical_clf loaded


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ model_ling loaded
✅ predict_email defined
✅ fuse_scores defined
✅ generate_explanation defined
✅ full_threat_scan defined

  ✅ ALL FUNCTIONS RESTORED — Ready for Phase 7 tests


In [ ]:
# ══════════════════════════════════════════════════════════════════
# FIXES FOR TEST 3 (False Positive) + TEST 4 (Override logic)
# Replace your existing fuse_scores and full_threat_scan definitions
# ══════════════════════════════════════════════════════════════════

# ── FIX 1: Trusted domain whitelist ──────────────────────────────
# Prevents the lexical engine from flagging well-known safe domains
TRUSTED_DOMAINS = {
    "google.com", "gmail.com", "youtube.com", "microsoft.com",
    "outlook.com", "apple.com", "amazon.com", "netflix.com",
    "github.com", "wikipedia.org", "stackoverflow.com",
    "linkedin.com", "twitter.com", "facebook.com", "instagram.com"
}

def is_trusted_domain(url: str) -> bool:
    """Returns True if the URL belongs to a known trusted domain."""
    try:
        import tldextract
        ext = tldextract.extract(str(url))
        registered = f"{ext.domain}.{ext.suffix}".lower()
        return registered in TRUSTED_DOMAINS
    except Exception:
        return False

print("✅ Trusted domain whitelist defined")

# ── FIX 2: Relaxed override rule (SPF + DKIM alone is enough) ────
def fuse_scores(s_ling=None, s_lex=None, s_vis=None,
                spf_fail=False, dkim_fail=False,
                blacklisted=False) -> dict:
    W = {"ling": 0.40, "lex": 0.35, "vis": 0.25}

    # Hard override rules — fire before soft ensemble
    # FIX: SPF + DKIM failure alone is strong enough signal
    if spf_fail and dkim_fail:
        return {"verdict": "Phishing", "score": 1.0,
                "override": True, "active_engines": ["hard_rules"]}

    if blacklisted:
        return {"verdict": "Phishing", "score": 1.0,
                "override": True, "active_engines": ["hard_rules"]}

    if s_vis is not None and s_vis > 0.85:
        return {"verdict": "Phishing", "score": 1.0,
                "override": True, "active_engines": ["visual"]}

    # Soft weighted ensemble
    scores  = {k: v for k, v in [("ling", s_ling),
                                  ("lex",  s_lex),
                                  ("vis",  s_vis)] if v is not None}
    if not scores:
        return {"verdict": "Unknown", "score": 0.0,
                "override": False, "active_engines": []}

    total_w = sum(W[k] for k in scores)
    s_total = sum(scores[k] * W[k] / total_w for k in scores)

    verdict = ("Phishing" if s_total >= 0.65 else
               "Scam"     if s_total >= 0.35 else
               "Genuine")

    return {"verdict": verdict, "score": round(s_total, 4),
            "override": False, "active_engines": list(scores.keys())}

print("✅ fuse_scores fixed (override now fires on SPF+DKIM alone)")

# ── FIX 3: full_threat_scan with whitelist check ─────────────────
def full_threat_scan(email_text=None, url_input=None, image_path=None,
                     brand_to_check=None, spf_fail=False, dkim_fail=False,
                     verbose=True) -> dict:

    s_ling = s_lex = s_vis = None
    clip_result = None
    flags = []

    # Whitelist check — skip lexical engine for trusted domains
    url_is_trusted = url_input and is_trusted_domain(url_input)
    if url_is_trusted:
        flags.append("TRUSTED_DOMAIN")

    # Engine 1: Linguistic (DistilBERT)
    if email_text:
        lr = predict_email(email_text)
        s_ling = min(lr["phishing"] + 0.5 * lr["spam"], 1.0)

    # Engine 2: Lexical (Random Forest)
    # Skip entirely if domain is trusted — prevents false positives
    if url_input and not url_is_trusted:
        feats = np.array(list(extract_url_features(url_input).values()))
        s_lex = float(lexical_clf.predict_proba([feats])[0][1])

    # Engine 3: Visual (CLIP)
    if image_path and brand_to_check:
        clip_result = predict_logo(image_path, brand_to_check)
        s_vis = clip_result["visual_phishing_score"]

    # Authentication flags
    if spf_fail:  flags.append("SPF_FAIL")
    if dkim_fail: flags.append("DKIM_FAIL")

    # Fuse scores
    verdict = fuse_scores(s_ling, s_lex, s_vis, spf_fail, dkim_fail)
    explanation = generate_explanation(None, None, clip_result, flags)

    score = verdict["score"]
    conf  = ("High"   if score >= 0.80 or verdict["override"] else
             "Medium" if score >= 0.50 else "Low")

    result = {
        **verdict, "confidence": conf,
        "scores": {
            "linguistic": round(s_ling, 4) if s_ling is not None else None,
            "lexical":    round(s_lex,  4) if s_lex  is not None else None,
            "visual":     round(s_vis,  4) if s_vis  is not None else None,
        },
        "flags": flags,
        "explanation": explanation
    }

    if verbose:
        v    = result["verdict"]
        icon = "🚨" if v == "Phishing" else ("⚠️" if v == "Scam" else "✅")
        print("\n" + "═" * 55)
        print(f"  {icon}  VERDICT: {v.upper()}  [{conf} — {score*100:.1f}%]")
        if url_is_trusted:
            print(f"  ℹ️  Trusted domain detected — Lexical engine skipped")
        print("═" * 55)
        for eng, sc in result["scores"].items():
            if sc is not None:
                bar = "█" * int(sc * 20)
                print(f"  {eng.capitalize():15s}: {sc:.4f}  {bar}")
        if flags:
            print(f"\n  Flags: {', '.join(flags)}")
        if explanation["top_features"]:
            print("\n  Evidence:")
            for f in explanation["top_features"]:
                print(f"    • {f}")
        print("═" * 55)

    return result

print("✅ full_threat_scan fixed (whitelist + corrected override)")
print("\n" + "="*55)
print("  Re-run Phase 7 test cases now")
print("="*55)

✅ Trusted domain whitelist defined
✅ fuse_scores fixed (override now fires on SPF+DKIM alone)
✅ full_threat_scan fixed (whitelist + corrected override)

  Re-run Phase 7 test cases now


In [ ]:
# ── PASTE INTO COLAB CELL 7-A ──────────────────────────────────────
# Environment: Google Colab
# Time: ~1 minute

print("="*60)
print("TEST 1 — Classic phishing email + malicious URL")
print("Expected: PHISHING (High confidence)")
full_threat_scan(
    email_text="URGENT: Your Netflix subscription has expired. Verify now.",
    url_input="http://netflix-billing-secure.xyz/login"
)

print("="*60)
print("TEST 2 — Tax refund scam")
print("Expected: PHISHING or SCAM")
full_threat_scan(
    email_text="Your tax refund of $450 is ready. Claim within 24 hours.",
    url_input="http://irs-refund-claim.net/verify?id=abc123"
)

print("="*60)
print("TEST 3 — Legitimate email")
print("Expected: GENUINE")
full_threat_scan(
    email_text="Hi, are we still meeting for coffee at 3pm?",
    url_input="https://www.google.com/maps"
)

print("="*60)
print("TEST 4 — Hard override: SPF + DKIM failure")
print("Expected: PHISHING (override=True)")
full_threat_scan(
    email_text="Verify your PayPal account.",
    url_input="http://paypa1-secure.biz/confirm",
    spf_fail=True, dkim_fail=True
)


TEST 1 — Classic phishing email + malicious URL
Expected: PHISHING (High confidence)

═══════════════════════════════════════════════════════
  🚨  VERDICT: PHISHING  [High — 83.7%]
═══════════════════════════════════════════════════════
  Linguistic     : 0.7000  ██████████████
  Lexical        : 0.9933  ███████████████████
═══════════════════════════════════════════════════════
TEST 2 — Tax refund scam
Expected: PHISHING or SCAM

═══════════════════════════════════════════════════════
  🚨  VERDICT: PHISHING  [High — 95.9%]
═══════════════════════════════════════════════════════
  Linguistic     : 0.9660  ███████████████████
  Lexical        : 0.9500  ███████████████████
═══════════════════════════════════════════════════════
TEST 3 — Legitimate email
Expected: GENUINE

═══════════════════════════════════════════════════════
  ✅  VERDICT: GENUINE  [Low — 6.5%]
  ℹ️  Trusted domain detected — Lexical engine skipped
═══════════════════════════════════════════════════════
  Linguistic    

{'verdict': 'Phishing',
 'score': 1.0,
 'override': True,
 'active_engines': ['hard_rules'],
 'confidence': 'High',
 'scores': {'linguistic': 0.5923, 'lexical': 1.0, 'visual': None},
 'flags': ['SPF_FAIL', 'DKIM_FAIL'],
 'explanation': {'top_features': ['Auth failure: SPF_FAIL',
   'Auth failure: DKIM_FAIL']}}

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 8-A (FIXED) — MLflow Experiment Tracking
# Fully self-contained — no dependency on previous cells
# ══════════════════════════════════════════════════════════════════
import mlflow, mlflow.sklearn, joblib, os, math, re
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import tldextract
from urllib.parse import urlparse

BASE = "/content/drive/MyDrive/Phishing_Project"

# ── Step 1: Redefine extract_url_features ────────────────────────
def extract_url_features(url: str) -> dict:
    url = str(url).strip()
    try:
        parsed = urlparse(url)
        ext    = tldextract.extract(url)
        host   = parsed.hostname or ""
        path   = parsed.path or ""
        query  = parsed.query or ""
        chars  = set(url)
        entropy = -sum(
            (url.count(c)/len(url)) * math.log2(url.count(c)/len(url))
            for c in chars if url.count(c) > 0
        ) if len(url) > 0 else 0
        SUSPICIOUS_WORDS = [
            "login","verify","secure","update","account","banking",
            "confirm","password","signin","paypal","ebay","amazon",
            "apple","microsoft","support","service","free","lucky"
        ]
        suspicious_count = sum(1 for w in SUSPICIOUS_WORDS if w in url.lower())
        digit_ratio = sum(c.isdigit() for c in host) / len(host) if host else 0
        return {
            "url_length":        len(url),
            "hostname_length":   len(host),
            "num_dots":          url.count("."),
            "has_at_symbol":     int("@" in url),
            "has_ip":            int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host))),
            "subdomain_depth":   len(ext.subdomain.split(".")) if ext.subdomain else 0,
            "entropy":           round(entropy, 4),
            "is_https":          int(parsed.scheme == "https"),
            "path_length":       len(path),
            "num_hyphens":       url.count("-"),
            "num_slashes":       url.count("/"),
            "num_query_params":  len(query.split("&")) if query else 0,
            "has_port":          int(bool(parsed.port)),
            "tld_suspicious":    int(ext.suffix in [
                                     "xyz","tk","ml","ga","cf","gq",
                                     "top","click","link","work","party"]),
            "suspicious_words":  suspicious_count,
            "digit_ratio_host":  round(digit_ratio, 4),
            "url_has_redirect":  int("redirect" in url.lower() or "url=" in url.lower()),
            "double_slash_path": int("//" in path),
        }
    except Exception:
        return {k: 0 for k in [
            "url_length","hostname_length","num_dots","has_at_symbol","has_ip",
            "subdomain_depth","entropy","is_https","path_length","num_hyphens",
            "num_slashes","num_query_params","has_port","tld_suspicious",
            "suspicious_words","digit_ratio_host","url_has_redirect","double_slash_path"
        ]}

print("✅ extract_url_features defined")

# ── Step 2: Reload dataset and recreate X_test, y_test ───────────
print("Loading URL dataset from Drive...")
url_df = pd.read_csv(f"{BASE}/Data/malicious_phish.csv")
url_df["label"] = url_df["type"].map({
    "benign": 0, "defacement": 1, "phishing": 1, "malware": 1
})
url_df = url_df[["url", "label"]].dropna()

# Same balancing as Phase 2-B
safe_df      = url_df[url_df["label"] == 0]
malicious_df = url_df[url_df["label"] == 1]
malicious_upsampled = resample(
    malicious_df, replace=True,
    n_samples=len(safe_df), random_state=42
)
url_balanced = pd.concat([safe_df, malicious_upsampled]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print("Extracting features for test split (this takes ~5 min)...")
X = pd.DataFrame([extract_url_features(u) for u in url_balanced["url"]])
y = url_balanced["label"].reset_index(drop=True)

# Same split as Phase 2-B — identical random_state reproduces same test set
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ X_test recreated: {X_test.shape}")

# ── Step 3: Reload trained model from Drive ───────────────────────
lexical_clf = joblib.load(f"{BASE}/Models/lexical_rf.pkl")
print("✅ lexical_clf loaded from Drive")

# ── Step 4: Log to MLflow ─────────────────────────────────────────
mlflow.set_tracking_uri(f"file://{BASE}/mlruns")
mlflow.set_experiment("phishing_detection")

with mlflow.start_run(run_name="lexical_rf_v2"):
    # Log parameters
    mlflow.log_param("n_estimators",  300)
    mlflow.log_param("class_weight",  "balanced")
    mlflow.log_param("feature_set",   "engineered_18_features")
    mlflow.log_param("test_size",     0.2)
    mlflow.log_param("label_mapping", "defacement=malicious")

    # Evaluate
    y_pred = lexical_clf.predict(X_test)
    f1  = f1_score(y_test, y_pred, average="weighted")
    acc = accuracy_score(y_test, y_pred)

    # Log metrics
    mlflow.log_metric("f1_weighted", f1)
    mlflow.log_metric("accuracy",    acc)

    # Log model artifact
    mlflow.sklearn.log_model(lexical_clf, "lexical_rf_model")

    print(f"\n📊 MLflow Run Logged:")
    print(f"   F1  (weighted) : {f1:.4f}")
    print(f"   Accuracy       : {acc:.4f}")
    print(f"   Saved to       : {BASE}/mlruns/")

print("\n✅ Phase 8-A complete.")

✅ extract_url_features defined
Loading URL dataset from Drive...
Extracting features for test split (this takes ~5 min)...
✅ X_test recreated: (171242, 18)
✅ lexical_clf loaded from Drive


2026/04/13 09:23:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 09:23:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



📊 MLflow Run Logged:
   F1  (weighted) : 0.9629
   Accuracy       : 0.9629
   Saved to       : /content/drive/MyDrive/Phishing_Project/mlruns/

✅ Phase 8-A complete.


In [ ]:
# ── PASTE INTO COLAB CELL 9-A ──────────────────────────────────────
# Environment: Google Colab
# Time: ~1 minute to start
# NOTE: Server runs in a background thread — cell completes but server keeps running

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional
import uvicorn, nest_asyncio, threading

app = FastAPI(title="Phishing Detection API", version="2.0")

class ScanRequest(BaseModel):
    email_body:     Optional[str]  = None
    url:            Optional[str]  = None
    brand_to_check: Optional[str]  = None
    spf_fail:       Optional[bool] = False
    dkim_fail:      Optional[bool] = False

@app.post("/scan")
async def scan(req: ScanRequest):
    return full_threat_scan(
        email_text=req.email_body, url_input=req.url,
        brand_to_check=req.brand_to_check,
        spf_fail=req.spf_fail or False,
        dkim_fail=req.dkim_fail or False,
        verbose=False
    )

@app.get("/health")
async def health(): return {"status": "ok", "version": "2.0"}

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, port=8000), daemon=True).start()
print("🚀 API server running on http://localhost:8000")
print("📖 Interactive docs: http://localhost:8000/docs")


🚀 API server running on http://localhost:8000
📖 Interactive docs: http://localhost:8000/docs


In [ ]:
# ── PASTE INTO COLAB CELL 10-A ──────────────────────────────────────
# Environment: Google Colab
# Time: instant

import os

print("=== FILES IN Google Drive Phishing_Project/ ===")
for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)
    indent = "" * level
    folder = os.path.basename(root)
    print(f"{indent}{folder}/")
    subindent = "" * (level + 1)
    for f in files:
        fpath = os.path.join(root, f)
        size  = os.path.getsize(fpath)
        size_str = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size//1024} KB"
        print(f"{subindent}{f}  ({size_str})")

print("\n✅ Verify these files exist:")
print("  ✓ Models/lexical_rf.pkl           (~20 MB)")
print("  ✓ Models/ling_model/pytorch_model.bin  (~260 MB)")
print("  ✓ Data/master_emails.csv")
print("  ✓ Data/malicious_phish.csv")



=== FILES IN Google Drive Phishing_Project/ ===
Phishing_Project/
Notebook/
Phishing_Detection_Improved.ipynb  (46 KB)
Data/
master_emails.csv  (155.3 MB)
malicious_phish.csv  (45.7 MB)
Models/
lexical_rf.pkl  (1278.3 MB)
url_feature_names.pkl  (0 KB)
ling_model/
config.json  (0 KB)
model.safetensors  (267.8 MB)
training_args.bin  (5 KB)
tokenizer_config.json  (0 KB)
tokenizer.json  (694 KB)
checkpoint-1313/
config.json  (0 KB)
model.safetensors  (267.8 MB)
training_args.bin  (5 KB)
optimizer.pt  (535.7 MB)
scheduler.pt  (1 KB)
scaler.pt  (1 KB)
rng_state.pth  (14 KB)
trainer_state.json  (3 KB)
checkpoint-2626/
config.json  (0 KB)
model.safetensors  (267.8 MB)
training_args.bin  (5 KB)
optimizer.pt  (535.7 MB)
scheduler.pt  (1 KB)
scaler.pt  (1 KB)
rng_state.pth  (14 KB)
trainer_state.json  (6 KB)
checkpoint-3939/
config.json  (0 KB)
model.safetensors  (267.8 MB)
training_args.bin  (5 KB)
optimizer.pt  (535.7 MB)
scheduler.pt  (1 KB)
scaler.pt  (1 KB)
rng_state.pth  (14 KB)
trainer_sta

In [ ]:
# ── Fix lexical_rf.pkl size ───────────────────────────────────────
import joblib, os

BASE = "/content/drive/MyDrive/Phishing_Project"

# Reload and re-save with maximum compression
lexical_clf = joblib.load(f"{BASE}/Models/lexical_rf.pkl")
joblib.dump(lexical_clf, f"{BASE}/Models/lexical_rf.pkl",
            compress=("zlib", 9))  # maximum compression

size = os.path.getsize(f"{BASE}/Models/lexical_rf.pkl") / 1e6
print(f"✅ lexical_rf.pkl compressed to: {size:.1f} MB")

✅ lexical_rf.pkl compressed to: 233.0 MB


In [ ]:
# ── Delete unnecessary checkpoint folders ────────────────────────
import shutil, os

BASE = "/content/drive/MyDrive/Phishing_Project"
ling_path = f"{BASE}/Models/ling_model"

checkpoints = [
    f"{ling_path}/checkpoint-1313",
    f"{ling_path}/checkpoint-2626",
    f"{ling_path}/checkpoint-3939",
]

for cp in checkpoints:
    if os.path.exists(cp):
        shutil.rmtree(cp)
        print(f"🗑️  Deleted: {cp}")
    else:
        print(f"⚠️  Not found (already deleted): {cp}")

print("\n✅ Checkpoints cleaned. Only best model kept.")

🗑️  Deleted: /content/drive/MyDrive/Phishing_Project/Models/ling_model/checkpoint-1313
🗑️  Deleted: /content/drive/MyDrive/Phishing_Project/Models/ling_model/checkpoint-2626
🗑️  Deleted: /content/drive/MyDrive/Phishing_Project/Models/ling_model/checkpoint-3939

✅ Checkpoints cleaned. Only best model kept.


In [ ]:
# ── Delete MLflow duplicate model artifact ────────────────────────
import shutil, os, glob

BASE = "/content/drive/MyDrive/Phishing_Project"

# Find and delete all model.pkl files inside mlruns (these are duplicates)
mlflow_models = glob.glob(f"{BASE}/mlruns/**/model.pkl", recursive=True)

for path in mlflow_models:
    size = os.path.getsize(path) / 1e6
    os.remove(path)
    print(f"🗑️  Deleted {path}  ({size:.1f} MB freed)")

if not mlflow_models:
    print("No MLflow model.pkl duplicates found.")

print("✅ MLflow artifacts cleaned.")

🗑️  Deleted /content/drive/MyDrive/Phishing_Project/mlruns/618052279499374712/models/m-11c6a893e1db493e8ed8ba1e5cb98fdc/artifacts/model.pkl  (1278.3 MB freed)
✅ MLflow artifacts cleaned.


In [ ]:
# ── Final storage check ───────────────────────────────────────────
import os

BASE = "/content/drive/MyDrive/Phishing_Project"

checks = {
    "lexical_rf.pkl":            f"{BASE}/Models/lexical_rf.pkl",
    "ling_model/model.safetensors": f"{BASE}/Models/ling_model/model.safetensors",
    "ling_model/tokenizer.json": f"{BASE}/Models/ling_model/tokenizer.json",
    "master_emails.csv":         f"{BASE}/Data/master_emails.csv",
    "malicious_phish.csv":       f"{BASE}/Data/malicious_phish.csv",
}

print("=== FINAL FILE CHECK ===\n")
total = 0
for name, path in checks.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        total += size
        status = "✅"
        # Warn if lexical model still too large
        if "lexical" in name and size > 100:
            status = "⚠️  Still large — re-run compression"
        print(f"  {status}  {name:45s} {size:8.1f} MB")
    else:
        print(f"  ❌  {name:45s} MISSING")

# Check no checkpoints remain
cp_found = False
for i in [1313, 2626, 3939]:
    cp = f"{BASE}/Models/ling_model/checkpoint-{i}"
    if os.path.exists(cp):
        print(f"  ⚠️  Checkpoint still exists: checkpoint-{i} — delete it")
        cp_found = True
if not cp_found:
    print(f"  ✅  {'No checkpoint folders remaining':45s}")

print(f"\n  Total essential storage: {total:.1f} MB")
print("\n=== PROJECT COMPLETE ✅ ===")

=== FINAL FILE CHECK ===

  ⚠️  Still large — re-run compression  lexical_rf.pkl                                   233.0 MB
  ✅  ling_model/model.safetensors                     267.8 MB
  ✅  ling_model/tokenizer.json                          0.7 MB
  ✅  master_emails.csv                                155.3 MB
  ✅  malicious_phish.csv                               45.7 MB
  ✅  No checkpoint folders remaining              

  Total essential storage: 702.6 MB

=== PROJECT COMPLETE ✅ ===


In [ ]:
# ══════════════════════════════════════════════════════════════════
# FINAL COMPLETE GRADIO APP — With CLIP + XAI (SHAP + LIME)
# ══════════════════════════════════════════════════════════════════
!pip install gradio shap lime tldextract -q

import gradio as gr
import torch, clip, shap, joblib, re, math
import numpy as np, pandas as pd
import transformers
from PIL import Image
from urllib.parse import urlparse
import tldextract, lime.lime_tabular as lt
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE   = "/content/drive/MyDrive/Phishing_Project"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load all models ───────────────────────────────────────────────
lexical_clf    = joblib.load(f"{BASE}/Models/lexical_rf.pkl")
tokenizer_ling = AutoTokenizer.from_pretrained(f"{BASE}/Models/ling_model")
model_ling     = AutoModelForSequenceClassification.from_pretrained(
                     f"{BASE}/Models/ling_model").to(device)
model_ling.eval()
clip_model, clip_prep = clip.load("ViT-B/32", device=device)
clip_model.eval()

FEATURE_ORDER = ["url_length","hostname_length","num_dots","has_at_symbol","has_ip",
                 "subdomain_depth","entropy","is_https","path_length","num_hyphens",
                 "num_slashes","num_query_params","has_port","tld_suspicious",
                 "suspicious_words","digit_ratio_host","url_has_redirect","double_slash_path"]

TRUSTED = {"google.com","gmail.com","youtube.com","microsoft.com","github.com",
           "amazon.com","apple.com","netflix.com","linkedin.com","wikipedia.org"}

def extract_features(url_str):
    parsed = urlparse(url_str); ext = tldextract.extract(url_str)
    host = parsed.hostname or ""; path = parsed.path or ""; query = parsed.query or ""
    chars = set(url_str)
    entropy = -sum((url_str.count(c)/len(url_str))*math.log2(url_str.count(c)/len(url_str))
                   for c in chars if url_str.count(c)>0) if len(url_str)>0 else 0
    sw = ["login","verify","secure","update","account","banking","confirm","password",
          "signin","paypal","ebay","amazon","apple","microsoft","support","service"]
    digit_ratio = sum(c.isdigit() for c in host)/len(host) if host else 0
    return {"url_length":len(url_str),"hostname_length":len(host),"num_dots":url_str.count("."),
            "has_at_symbol":int("@" in url_str),"has_ip":int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$",host))),
            "subdomain_depth":len(ext.subdomain.split(".")) if ext.subdomain else 0,
            "entropy":round(entropy,4),"is_https":int(parsed.scheme=="https"),
            "path_length":len(path),"num_hyphens":url_str.count("-"),
            "num_slashes":url_str.count("/"),"num_query_params":len(query.split("&")) if query else 0,
            "has_port":int(bool(parsed.port)),"tld_suspicious":int(ext.suffix in ["xyz","tk","ml","ga","cf","top"]),
            "suspicious_words":sum(1 for w in sw if w in url_str.lower()),
            "digit_ratio_host":round(digit_ratio,4),
            "url_has_redirect":int("redirect" in url_str.lower() or "url=" in url_str.lower()),
            "double_slash_path":int("//" in path)}

# ── CLIP logo scorer ──────────────────────────────────────────────
def check_logo(image, brand_name):
    if image is None or not brand_name.strip():
        return None, "No image provided"
    try:
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        img_t = clip_prep(image.convert("RGB")).unsqueeze(0).to(device)
        prompts = clip.tokenize([f"official {brand_name} logo","unrelated graphic"]).to(device)
        with torch.no_grad():
            i_feat = clip_model.encode_image(img_t)
            t_feat = clip_model.encode_text(prompts)
            i_feat = i_feat / i_feat.norm(dim=-1, keepdim=True)
            t_feat = t_feat / t_feat.norm(dim=-1, keepdim=True)
            sim = (i_feat @ t_feat.T).squeeze()
        match = sim[0].item()
        impersonation = 1.0 - match
        verdict = "Logo mismatch — impersonation likely" if match < 0.5 else "Logo matches claimed brand"
        detail = f"Brand match score: {match:.3f} | Impersonation score: {impersonation:.3f} | {verdict}"
        return impersonation, detail
    except Exception as e:
        return None, f"CLIP error: {e}"

# ── SHAP for email ────────────────────────────────────────────────
def get_shap_explanation(text):
    try:
        pipe = transformers.pipeline("text-classification", model=model_ling,
                                     tokenizer=tokenizer_ling, return_all_scores=True,
                                     device=0 if torch.cuda.is_available() else -1)
        explainer = shap.Explainer(pipe)
        sv = explainer([text])
        pred_class = int(np.argmax([sv.values[0,:,i].sum() for i in range(3)]))
        pairs = list(zip(sv.data[0], sv.values[0,:,pred_class].tolist()))
        top = sorted(pairs, key=lambda x: abs(x[1]), reverse=True)[:5]
        return [f'"{tok}" (impact {score:+.3f})' for tok, score in top if abs(score) > 0.03]
    except Exception:
        return []

# ── LIME for URL ──────────────────────────────────────────────────
_lime_explainer = None
def get_lime_explanation(url_str):
    global _lime_explainer
    try:
        if _lime_explainer is None:
            dummy = np.zeros((10, len(FEATURE_ORDER)))
            _lime_explainer = lt.LimeTabularExplainer(
                dummy, feature_names=FEATURE_ORDER,
                class_names=["Safe","Malicious"], mode="classification")
        feats = np.array(list(extract_features(url_str).values()))
        exp = _lime_explainer.explain_instance(feats, lexical_clf.predict_proba, num_features=4)
        return [f"{f}: {s:+.3f}" for f, s in exp.as_list() if s > 0.01]
    except Exception:
        return []

# ── Main scan function ────────────────────────────────────────────
def full_scan(email_text, url_input, image, brand_name, spf_fail, dkim_fail):
    s_ling = s_lex = s_vis = None
    evidence = []
    flags = []

    # Linguistic engine + SHAP
    if email_text and email_text.strip():
        inputs = tokenizer_ling(email_text, return_tensors="pt",
                                truncation=True, max_length=256).to(device)
        with torch.no_grad():
            probs = torch.softmax(model_ling(**inputs).logits, dim=-1)[0].tolist()
        s_ling = min(probs[2] + 0.5*probs[1], 1.0)
        shap_vals = get_shap_explanation(email_text)
        for sv in shap_vals:
            evidence.append(f"[SHAP] Suspicious token: {sv}")

    # Lexical engine + LIME
    if url_input and url_input.strip():
        url_str = url_input.strip()
        if not url_str.startswith(("http://","https://")):
            url_str = "http://" + url_str
        ext = tldextract.extract(url_str)
        domain = f"{ext.domain}.{ext.suffix}".lower()
        if domain in TRUSTED:
            flags.append("TRUSTED_DOMAIN")
            s_lex = 0.0
        else:
            feats = np.array(list(extract_features(url_str).values()))
            s_lex = float(lexical_clf.predict_proba([feats])[0][1])
            lime_vals = get_lime_explanation(url_str)
            for lv in lime_vals:
                evidence.append(f"[LIME] {lv}")

    # CLIP visual engine
    if image is not None and brand_name and brand_name.strip():
        s_vis, clip_detail = check_logo(image, brand_name)
        evidence.append(f"[CLIP] {clip_detail}")

    # Auth flags
    if spf_fail:  flags.append("SPF_FAIL");  evidence.append("[Auth] SPF authentication failed")
    if dkim_fail: flags.append("DKIM_FAIL"); evidence.append("[Auth] DKIM authentication failed")

    # Hard overrides
    if spf_fail and dkim_fail:
        verdict, score, conf = "PHISHING", 1.0, "High (override)"
    elif s_vis is not None and s_vis > 0.85:
        verdict, score, conf = "PHISHING", 1.0, "High (visual override)"
    else:
        # Weighted ensemble
        W = {"ling":0.40, "lex":0.35, "vis":0.25}
        scores = {k:v for k,v in [("ling",s_ling),("lex",s_lex),("vis",s_vis)] if v is not None}
        if not scores:
            return "No input provided","—","—","Please enter email, URL or image."
        tw = sum(W[k] for k in scores)
        score = sum(scores[k]*W[k]/tw for k in scores)
        verdict = "PHISHING" if score>=0.65 else ("SCAM" if score>=0.35 else "GENUINE")
        conf = "High" if score>=0.80 else ("Medium" if score>=0.50 else "Low")

    score_txt = f"{score*100:.1f}%"
    engines = (f"Linguistic: {s_ling*100:.1f}%  " if s_ling is not None else "") + \
              (f"Lexical: {s_lex*100:.1f}%  " if s_lex is not None else "") + \
              (f"Visual: {s_vis*100:.1f}%  " if s_vis is not None else "")
    flags_txt = ", ".join(flags) if flags else "None"
    evidence_txt = "\n".join(f"  • {e}" for e in evidence[:6]) if evidence else "  No specific indicators found."

    report = f"Flags: {flags_txt}\nEngine scores: {engines}\n\nEvidence (XAI):\n{evidence_txt}"
    return verdict, score_txt, conf, report

# ── Gradio UI ─────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="PhishGuard AI") as demo:
    gr.Markdown("# PhishGuard AI — Intelligent Phishing Detection")
    gr.Markdown("**DistilBERT** (Linguistic) + **Random Forest** (Lexical) + **CLIP** (Visual) + **SHAP/LIME** (XAI)")

    with gr.Row():
        with gr.Column(scale=1):
            email_input = gr.Textbox(label="Email body (optional)", lines=5,
                placeholder="Paste full email text here...")
            url_input   = gr.Textbox(label="URL to check (optional)",
                placeholder="http://paypa1-secure.xyz/login")
            image_input = gr.Image(label="Screenshot / logo image (optional)", type="pil")
            brand_input = gr.Textbox(label="Claimed brand name (for CLIP)",
                placeholder="e.g. PayPal, Netflix, Microsoft")
            with gr.Row():
                spf_box  = gr.Checkbox(label="SPF failure detected")
                dkim_box = gr.Checkbox(label="DKIM failure detected")
            scan_btn = gr.Button("Run Full Scan", variant="primary", size="lg")

        with gr.Column(scale=1):
            verdict_out  = gr.Textbox(label="Verdict", lines=1)
            score_out    = gr.Textbox(label="Threat score", lines=1)
            conf_out     = gr.Textbox(label="Confidence", lines=1)
            report_out   = gr.Textbox(label="XAI Report — Evidence & Flags", lines=12)

    gr.Examples(
        examples=[
            ["URGENT: Your Netflix account has been suspended. Verify your payment now.", "http://netflix-billing-secure.xyz/login", None, "Netflix", False, False],
            ["Hi, are we still meeting for coffee tomorrow?", "https://www.google.com/maps", None, "", False, False],
            ["Your tax refund of $450 is ready. Claim within 24 hours.", "http://irs-refund-claim.net/verify", None, "", True, True],
        ],
        inputs=[email_input, url_input, image_input, brand_input, spf_box, dkim_box]
    )

    scan_btn.click(fn=full_scan,
                   inputs=[email_input, url_input, image_input, brand_input, spf_box, dkim_box],
                   outputs=[verdict_out, score_out, conf_out, report_out])

demo.launch(share=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_1076/849936198.py:180: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="PhishGuard AI") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bad15bba90f10bfa87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import json

# Load the broken notebook
with open("/content/drive/MyDrive/Colab Notebooks/Training_Project.ipynb", "r") as f:
    nb = json.load(f)

# Remove broken widget metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]
    print("Widgets metadata removed!")
else:
    print("No widgets metadata found.")

# Save the fixed notebook
with open("/content/drive/MyDrive/Colab Notebooks/Training_Project_fixed.ipynb", "w") as f:
    json.dump(nb, f, indent=2)

print("Done! Fixed file saved.")

Widgets metadata removed!
Done! Fixed file saved.


In [4]:
from google.colab import files
files.download("/content/drive/MyDrive/Colab Notebooks/Training_Project_fixed.ipynb")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>